# Notebook 02 — EEG Data Loading & Preprocessing
## Dataset ds005284 · Zhao et al. · OpenNeuro (CC0)
Real BDF files are loaded from `data/ds005284/` and processed with MNE.

In [1]:
import warnings, pathlib, json
warnings.filterwarnings("ignore")
import mne, numpy as np, pandas as pd
from scipy.integrate import trapezoid
mne.set_log_level("ERROR")

BIDS        = pathlib.Path("data/ds005284")
TASK        = "26ByBiosemi"
SFREQ_OUT   = 256.0
S1_CHS      = ["C3", "CZ", "C4"]
PAIN_TMIN, PAIN_TMAX = -0.2, 2.0
BASE_ANCHOR_OFFSET   = -5.0   # seconds before pain onset
BASE_DUR             = 2.0    # seconds

ptable = pd.read_csv(BIDS / "participants.tsv", sep="\t")
print(f"Participants file: {BIDS/'participants.tsv'}")
print(f"Subjects found:    {len(ptable)}")
print()
print(ptable[["participant_id","Gender","Age","Pain_Threshold_7"]].to_string(index=False))

Participants file: data/ds005284/participants.tsv
Subjects found:    26

participant_id Gender  Age  Pain_Threshold_7
       sub-001      M   25              3.50
       sub-002      F   22              3.50
       sub-003      F   21              3.75
       sub-004      M   22              3.25
       sub-005      F   23              3.00
       sub-006      F   19              4.25
       sub-007      M   23              3.75
       sub-008      M   19              4.25
       sub-009      F   20              3.50
       sub-010      F   23              2.75
       sub-011      F   22              3.25
       sub-012      F   18              3.50
       sub-013      F   19              4.25
       sub-014      M   23              3.50
       sub-015      M   24              3.75
       sub-016      M   22              3.50
       sub-017      F   19              3.50
       sub-018      M   24              3.50
       sub-019      F   19              4.50
       sub-020      F   20 

In [2]:
# Verify BDF files exist and show sizes
print(f"{'Subject':12s}  {'BDF file':55s}  {'Size (MB)':>9s}")
print("-" * 82)
for sub in sorted(ptable["participant_id"]):
    bdf = BIDS / sub / "eeg" / f"{sub}_task-{TASK}_eeg.bdf"
    ev  = BIDS / sub / "eeg" / f"{sub}_task-{TASK}_events.tsv"
    size_mb = bdf.stat().st_size / 1e6 if bdf.exists() else 0
    print(f"{sub:12s}  {str(bdf):55s}  {size_mb:9.1f}")
print()
total_gb = sum(
    (BIDS/s/"eeg"/f"{s}_task-{TASK}_eeg.bdf").stat().st_size
    for s in ptable["participant_id"]
    if (BIDS/s/"eeg"/f"{s}_task-{TASK}_eeg.bdf").exists()
) / 1e9
print(f"Total BDF data on disk: {total_gb:.2f} GB")

Subject       BDF file                                                 Size (MB)
----------------------------------------------------------------------------------
sub-001       data/ds005284/sub-001/eeg/sub-001_task-26ByBiosemi_eeg.bdf       65.5
sub-002       data/ds005284/sub-002/eeg/sub-002_task-26ByBiosemi_eeg.bdf       65.7
sub-003       data/ds005284/sub-003/eeg/sub-003_task-26ByBiosemi_eeg.bdf       65.9
sub-004       data/ds005284/sub-004/eeg/sub-004_task-26ByBiosemi_eeg.bdf       65.1
sub-005       data/ds005284/sub-005/eeg/sub-005_task-26ByBiosemi_eeg.bdf       69.1
sub-006       data/ds005284/sub-006/eeg/sub-006_task-26ByBiosemi_eeg.bdf       66.1
sub-007       data/ds005284/sub-007/eeg/sub-007_task-26ByBiosemi_eeg.bdf       65.3
sub-008       data/ds005284/sub-008/eeg/sub-008_task-26ByBiosemi_eeg.bdf       66.1
sub-009       data/ds005284/sub-009/eeg/sub-009_task-26ByBiosemi_eeg.bdf       66.3
sub-010       data/ds005284/sub-010/eeg/sub-010_task-26ByBiosemi_eeg.bdf       6

In [3]:
# Inspect raw BDF for sub-001 — show real channel names before renaming
raw_probe = mne.io.read_raw_bdf(
    str(BIDS/"sub-001"/"eeg"/"sub-001_task-26ByBiosemi_eeg.bdf"),
    preload=False, verbose=False)
print(f"Native BDF channel count:  {len(raw_probe.ch_names)}")
print(f"Native BDF sampling rate:  {raw_probe.info['sfreq']} Hz")
print(f"Recording duration:        {raw_probe.times[-1]:.1f} s")
print(f"Native channel names (first 10): {raw_probe.ch_names[:10]}")

ch_tsv = pd.read_csv(BIDS/"sub-001"/"eeg"/"sub-001_task-26ByBiosemi_channels.tsv", sep="\t")
print(f"\nchannels.tsv 10-20 names (first 10): {ch_tsv['name'].tolist()[:10]}")
print(f"Channel rename: A1→{ch_tsv['name'][0]}, A2→{ch_tsv['name'][1]}, "
      f"..., B16→{ch_tsv['name'][47]}")

Native BDF channel count:  65
Native BDF sampling rate:  1024.0 Hz


Recording duration:        328.0 s
Native channel names (first 10): ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10']

channels.tsv 10-20 names (first 10): ['FP1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7', 'FC5', 'FC3']
Channel rename: A1→FP1, A2→AF7, ..., B16→CZ


In [4]:
# Show events for sub-001 — real trigger codes from the BDF
ev_df = pd.read_csv(BIDS/"sub-001"/"eeg"/"sub-001_task-26ByBiosemi_events.tsv", sep="\t")
print(f"Events file: {BIDS/'sub-001'/'eeg'/'sub-001_task-26ByBiosemi_events.tsv'}")
print(f"Total events: {len(ev_df)}")
print()
print("Event value counts:")
print(ev_df["value"].value_counts().to_string())
print()
print("Pain onset times (condition 54, seconds from recording start):")
pain_times = ev_df[ev_df["value"]=="condition 54"]["onset"].values
print(pain_times)
print(f"\nISI (inter-stimulus interval): {np.diff(pain_times).mean():.2f} ± {np.diff(pain_times).std():.2f} s")

Events file: data/ds005284/sub-001/eeg/sub-001_task-26ByBiosemi_events.tsv
Total events: 30

Event value counts:
value
condition 54    16
condition 64    14

Pain onset times (condition 54, seconds from recording start):
[121.91503906 134.06542969 146.94824219 159.98144531 172.86523438
 185.59765625 198.26464844 211.39746094 224.48144531 236.93066406
 249.1640625  261.38085938 274.14746094 287.03027344 299.38085938
 312.48046875]

ISI (inter-stimulus interval): 12.70 ± 0.33 s


In [5]:
# Full preprocessing on sub-001 to demonstrate real pipeline output
import mne
sub = "sub-001"
bdf_path = BIDS/sub/"eeg"/f"{sub}_task-{TASK}_eeg.bdf"
ch_tsv   = BIDS/sub/"eeg"/f"{sub}_task-{TASK}_channels.tsv"

raw = mne.io.read_raw_bdf(str(bdf_path), preload=True, verbose=False)
print(f"Loaded: {bdf_path.name}")
print(f"  Raw channels: {len(raw.ch_names)}  |  Sfreq: {raw.info['sfreq']} Hz  |  Duration: {raw.times[-1]:.1f} s")

# Rename A1/B1/... → 10-20 via channels.tsv
ch_df   = pd.read_csv(ch_tsv, sep="\t")
std_nms = ch_df["name"].str.upper().tolist()
eeg_bdf = [c for c in raw.ch_names if c.upper() not in ("STATUS","TRIGGERS")]
raw.rename_channels({eeg_bdf[i]: std_nms[i] for i in range(min(len(eeg_bdf), len(std_nms)))})
raw.rename_channels({c: c.upper() for c in raw.ch_names})
for c in raw.ch_names:
    try: raw.set_channel_types({c: "stim" if c in ("STATUS","TRIGGERS") else "eeg"})
    except: pass

print(f"  After rename — C3 present: {'C3' in raw.ch_names}, CZ: {'CZ' in raw.ch_names}, C4: {'C4' in raw.ch_names}")

raw.resample(SFREQ_OUT, verbose=False)
print(f"  Downsampled to {raw.info['sfreq']} Hz  ({raw.n_times} samples)")

raw.notch_filter([50.0, 100.0], picks="eeg", verbose=False)
raw.filter(0.5, 80.0, picks="eeg", verbose=False)
print(f"  Notch 50/100 Hz + bandpass 0.5-80 Hz applied")

raw.set_eeg_reference("average", projection=False, verbose=False)
print(f"  Average reference applied")

# Epoch — pain
ev_df2   = pd.read_csv(BIDS/sub/"eeg"/f"{sub}_task-{TASK}_events.tsv", sep="\t")
pain_on  = ev_df2[ev_df2["value"]=="condition 54"]["onset"].values
sfreq    = raw.info["sfreq"]
p_samps  = (pain_on * sfreq).round().astype(int)
p_samps  = p_samps[(p_samps >= int(0.3*sfreq)) & (p_samps < raw.n_times - int((PAIN_TMAX+0.3)*sfreq))]
b_samps  = (pain_on * sfreq + BASE_ANCHOR_OFFSET * sfreq).round().astype(int)
b_samps  = b_samps[(b_samps >= 0) & (b_samps < raw.n_times - int((BASE_DUR+0.3)*sfreq))]
b_samps  = b_samps[:len(p_samps)]
p_samps  = p_samps[:len(b_samps)]

ev_pain = np.c_[p_samps, np.zeros(len(p_samps),int), np.ones(len(p_samps),int)]
ev_base = np.c_[b_samps, np.zeros(len(b_samps),int), np.full(len(b_samps),2,int)]

s1 = ["C3","CZ","C4"]
ep_pain = mne.Epochs(raw, ev_pain, event_id={"pain":1}, tmin=PAIN_TMIN, tmax=PAIN_TMAX,
                     baseline=(PAIN_TMIN, 0.0), picks=s1, reject=dict(eeg=200e-6),
                     preload=True, verbose=False)
ep_base = mne.Epochs(raw, ev_base, event_id={"nopain":2}, tmin=0.0, tmax=BASE_DUR,
                     baseline=None, picks=s1, reject=dict(eeg=200e-6),
                     preload=True, verbose=False)

print(f"\nEpochs for {sub}:")
print(f"  Pain epochs retained:    {len(ep_pain)} / {len(ev_pain)}")
print(f"  No-pain epochs retained: {len(ep_base)} / {len(ev_base)}")
print(f"  S1 channels: {ep_pain.ch_names}")
print(f"  Pain epoch shape: {ep_pain.get_data().shape}  (n_epochs × n_ch × n_times)")
print(f"  Mean CZ amplitude during pain:    {ep_pain.get_data()[:,ep_pain.ch_names.index('CZ'),:].mean()*1e6:.4f} µV")
print(f"  Mean CZ amplitude during no-pain: {ep_base.get_data()[:,ep_base.ch_names.index('CZ'),:].mean()*1e6:.4f} µV")

Loaded: sub-001_task-26ByBiosemi_eeg.bdf
  Raw channels: 65  |  Sfreq: 1024.0 Hz  |  Duration: 328.0 s
  After rename — C3 present: True, CZ: True, C4: True


  Downsampled to 256.0 Hz  (83968 samples)


  Notch 50/100 Hz + bandpass 0.5-80 Hz applied
  Average reference applied

Epochs for sub-001:
  Pain epochs retained:    14 / 16
  No-pain epochs retained: 16 / 16
  S1 channels: ['C3', 'CZ', 'C4']
  Pain epoch shape: (14, 3, 564)  (n_epochs × n_ch × n_times)
  Mean CZ amplitude during pain:    1.0507 µV
  Mean CZ amplitude during no-pain: 0.5313 µV


In [6]:
# Summary across all 26 subjects — read from the preprocessed subject_summary.csv
summary = pd.read_csv("data/preprocessed/subject_summary.csv")
print(f"Subject summary file: data/preprocessed/subject_summary.csv")
print(f"Subjects processed: {len(summary)}")
print()
print(summary[["subject","n_pain","n_nopain","pain_threshold"]].to_string(index=False))
print()
print(f"Total pain epochs:    {summary['n_pain'].sum()}")
print(f"Total no-pain epochs: {summary['n_nopain'].sum()}")
print(f"Overall total:        {summary['n_pain'].sum() + summary['n_nopain'].sum()}")

Subject summary file: data/preprocessed/subject_summary.csv
Subjects processed: 26

subject  n_pain  n_nopain  pain_threshold
sub-001      14        16            3.50
sub-002      16        16            3.50
sub-003      16        16            3.75
sub-004      15        16            3.25
sub-005      16        16            3.00
sub-006       8        15            4.25
sub-007      14        16            3.75
sub-008       7        16            4.25
sub-009      10        16            3.50
sub-010       9        16            2.75
sub-011      16        16            3.25
sub-012      12        16            3.50
sub-013      16        16            4.25
sub-014      15        16            3.50
sub-015      16        16            3.75
sub-016      16        16            3.50
sub-017      16        16            3.50
sub-018      15        16            3.50
sub-019      12        16            4.50
sub-020      16        16            3.75
sub-021      14        16         